# Build scenario-level dataset for scenario discovery

Reads the Scenario Compass Initiative pathways ensemble
(`data/SCI-2025_v1.0_pathways_ensemble_global.xlsx`) and produces one clean, scenario-level table
(one row per Model-Scenario) holding exactly what notebooks 02 and 03 need:

- the two `meta`-sheet fields that define the outcome (year of net-zero CO2, cumulative CCS)
- the **nine** pathway-descriptor factors we analyse, as full 2010-2100 trajectories so any snapshot
  year - we use **2030** and **2050** - is available. Seven come straight from the long `data` sheet;
  two are derived here: **Electrification** (final-energy electricity share) and
  **GDP per capita|PPP**.
- the binary outcome labels (`success_nz2070`, `low_ccs_reliance`, `desired_success`)

Output: `data_for_scenariodiscovery_full.csv`.

In [43]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

In [44]:
XLSX_PATH = "../../data/SCI-2025_v1.0_pathways_ensemble_global.xlsx"
ENGINE = "calamine"  # python-calamine: much faster than openpyxl on this 120MB file
OUTPUT_CSV = "data_for_scenariodiscovery_full.csv"

## 1. Load the `meta` sheet (outcome fields)

One row per Model-Scenario. We only need two fields from it, both used to build the outcome:
`Emissions Diagnostics|Year of Net Zero|CO2` (net-zero timing) and
`Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]` (CCS reliance).

In [45]:
meta = pd.read_excel(XLSX_PATH, sheet_name="meta", engine=ENGINE)
print(meta.shape)

NZ_COL = "Emissions Diagnostics|Year of Net Zero|CO2"
CCS_COL = "Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"
meta[[NZ_COL, CCS_COL]].describe()

(1599, 71)


,Emissions Diagnostics|Year of Net Zero|CO2,"Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"
count,909.000000,1509.000000
mean,2070.380638,522.127105
std,14.398448,399.460534
min,2030.000000,0.000000
25%,2060.000000,219.651632
50%,2070.000000,493.199712
75%,2080.000000,729.805161
max,2100.000000,2145.347648


## 2. Load the `data` sheet and pivot the raw variables to wide format

The `data` sheet is long IAMC format (Model, Scenario, Region, Variable, Unit, one column per year);
Region is always `World`. We pull the seven raw factors **plus** the three series needed to derive
the two extra factors (`Final Energy` total, `GDP|PPP`, `Population`), then pivot `Variable` into
columns suffixed by year (`<variable>|<year>`).

In [ ]:
RAW_VARS = [
    "Emissions|CO2",
    "Primary Energy|Fossil",
    "Primary Energy|Biomass",
    "Primary Energy|Non-Biomass Renewables",
    "Carbon Capture|Geological Storage|Biomass",
    "Final Energy|Electricity",
    "Final Energy",          # denominator for the electrification share
    "GDP|PPP",               # numerator for GDP per capita
    "Population",            # denominator for GDP per capita
    "Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]",
    "Gross Emissions|CO2|AFOLU",
]

data_long = pd.read_excel(XLSX_PATH, sheet_name="data", engine=ENGINE)
assert data_long["Region"].unique().tolist() == ["World"]

missing = set(RAW_VARS) - set(data_long["Variable"].unique())
assert not missing, f"Variables not found in data sheet: {missing}"

filtered = data_long[data_long["Variable"].isin(RAW_VARS)].copy()
year_cols = [c for c in filtered.columns if c.isdigit()]

wide = filtered.set_index(["Model", "Scenario", "Variable"])[year_cols].unstack("Variable")
wide.columns = [f"{var}|{year}" for year, var in wide.columns]
wide = wide.reset_index()
print(wide.shape)
wide.head()

(1599, 192)


,Model,Scenario,Carbon Capture|Geological Storage|Biomass|2010,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2010,Emissions|CO2|2010,Final Energy|2010,Final Energy|Electricity|2010,GDP|PPP|2010,Population|2010,Primary Energy|Biomass|2010,Primary Energy|Fossil|2010,Primary Energy|Non-Biomass Renewables|2010,Carbon Capture|Geological Storage|Biomass|2015,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2015,Emissions|CO2|2015,Final Energy|2015,Final Energy|Electricity|2015,GDP|PPP|2015,Population|2015,Primary Energy|Biomass|2015,Primary Energy|Fossil|2015,Primary Energy|Non-Biomass Renewables|2015,Carbon Capture|Geological Storage|Biomass|2020,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2020,Emissions|CO2|2020,Final Energy|2020,Final Energy|Electricity|2020,GDP|PPP|2020,Population|2020,Primary Energy|Biomass|2020,...,Carbon Capture|Geological Storage|Biomass|2090,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2090,Emissions|CO2|2090,Final Energy|2090,Final Energy|Electricity|2090,GDP|PPP|2090,Population|2090,Primary Energy|Biomass|2090,Primary Energy|Fossil|2090,Primary Energy|Non-Biomass Renewables|2090,Carbon Capture|Geological Storage|Biomass|2095,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2095,Emissions|CO2|2095,Final Energy|2095,Final Energy|Electricity|2095,GDP|PPP|2095,Population|2095,Primary Energy|Biomass|2095,Primary Energy|Fossil|2095,Primary Energy|Non-Biomass Renewables|2095,Carbon Capture|Geological Storage|Biomass|2100,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2100,Emissions|CO2|2100,Final Energy|2100,Final Energy|Electricity|2100,GDP|PPP|2100,Population|2100,Primary Energy|Biomass|2100,Primary Energy|Fossil|2100,Primary Energy|Non-Biomass Renewables|2100
0,AIM/CGE 2.0,SSP1-19,0.0,0.975563,35782.6485,342.2994,64.5446,71310.55217,6879.5896,44.9428,400.6963,13.3812,0.0,1.115697,36508.26915,354.12755,74.22065,89836.671385,7204.40845,43.70495,418.17680,17.40440,0.0,1.225605,37233.8898,365.9557,83.8967,108362.7906,7529.2273,42.4671,...,3164.8677,1.177504,-4390.4928,324.1516,248.5773,560556.7,7435.4936,86.6641,54.7800,240.2778,NaN,1.162928,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3246.0161,1.132365,-4474.8741,311.6341,243.8105,606277.1,6881.5846,86.7606,46.9562,236.0284
1,AIM/CGE 2.0,SSP1-26,0.0,0.975563,35781.8143,342.3496,64.5452,71311.61510,6879.5896,44.9429,400.7582,13.3809,0.0,1.115698,36481.57640,354.17240,74.23840,89838.073450,7204.40845,43.70375,418.27880,17.40285,0.0,1.225277,37181.3385,365.9952,83.9316,108364.5318,7529.2273,42.4646,...,2875.0978,1.629364,626.9675,340.2928,225.3647,563268.2,7435.4936,76.8439,171.6095,185.2776,NaN,1.631904,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3027.9907,1.609574,17.7377,326.1978,222.4464,608721.3,6881.5846,77.9287,164.1606,180.6178
2,AIM/CGE 2.0,SSP1-34,0.0,0.975563,35781.8143,342.3496,64.5452,71311.61510,6879.5896,44.9429,400.7582,13.3809,0.0,1.115698,36481.57640,354.17240,74.23840,89838.073450,7204.40845,43.70375,418.27880,17.40285,0.0,1.225268,37181.3385,365.9952,83.9316,108364.5318,7529.2273,42.4646,...,678.0178,2.071235,10882.1926,363.9752,211.4716,566955.4,7435.4936,85.0360,245.1817,151.4369,NaN,2.089055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,951.6729,2.090244,9037.7818,348.4575,212.5913,612448.1,6881.5846,87.8550,230.8559,151.7119
3,AIM/CGE 2.0,SSP1-45,NaN,0.975563,35781.8143,342.3496,64.5452,71311.61510,6879.5896,44.9429,400.7582,13.3809,NaN,1.115711,36185.22300,352.76025,74.17075,89820.027650,7204.40845,43.76595,415.62180,17.78775,NaN,1.226074,36588.6317,363.1709,83.7963,108328.4402,7529.2273,42.5890,...,NaN,2.620006,21544.6626,381.3087,206.3124,569001.4,7435.4936,73.9360,307.2986,129.4012,NaN,2.652736,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.664423,18738.1346,361.3381,207.8181,614156.4,6881.5846,76.3293,275.1111,134.0676
4,AIM/CGE 2.0,SSP1-Baseline,NaN,0.975563,35888.7492,342.5024,64.4927,71311.99680,6879.5896,44.9441,401.6235,13.2658,NaN,1.115659,37509.54180

## 3. Merge meta + pivoted factors

Left-join on `meta` (the master 1599-row Model-Scenario list) so a scenario missing one factor still
appears, just with NaNs for that column.

In [47]:
merged = meta.merge(wide, on=["Model", "Scenario"], how="left", validate="one_to_one")
print(merged.shape)

(1599, 261)


## 4. Derive the two extra factors

- **Electrification** = `Final Energy|Electricity` / `Final Energy` - the share of final energy
  delivered as electricity (dimensionless, ~0-1). Captures the *depth* of electrification rather
  than the absolute electricity level.
- **GDP per capita|PPP** = `GDP|PPP` (billion USD_2010/yr) / `Population` (million) x 1000, giving
  thousand USD_2010 per capita.

Both are derived for every year where the inputs are reported.

In [48]:
fe_years = sorted(
    int(c.rsplit("|", 1)[1]) for c in merged.columns
    if c.rsplit("|", 1)[0] == "Final Energy" and c.rsplit("|", 1)[1].isdigit()
)
for y in fe_years:
    elec, total = f"Final Energy|Electricity|{y}", f"Final Energy|{y}"
    if elec in merged and total in merged:
        merged[f"Electrification|{y}"] = merged[elec] / merged[total]

gdp_years = sorted(
    int(c.rsplit("|", 1)[1]) for c in merged.columns
    if c.rsplit("|", 1)[0] == "GDP|PPP" and c.rsplit("|", 1)[1].isdigit()
)
for y in gdp_years:
    merged[f"GDP per capita|PPP|{y}"] = merged[f"GDP|PPP|{y}"] / merged[f"Population|{y}"] * 1000

print(f"Electrification derived for {len(fe_years)} years; "
      f"2050 non-null = {merged['Electrification|2050'].notna().sum()}, "
      f"range {merged['Electrification|2050'].min():.2f}-{merged['Electrification|2050'].max():.2f}")
print(f"GDP per capita derived for {len(gdp_years)} years; "
      f"2050 non-null = {merged['GDP per capita|PPP|2050'].notna().sum()}")

Electrification derived for 19 years; 2050 non-null = 1557, range 0.18-0.74
GDP per capita derived for 19 years; 2050 non-null = 1304


## 5. Build the outcome: net zero by 2070 *and* low CCS reliance

- `success_nz2070`: net-zero CO2 reached at or before 2070 (NaN year = never within horizon -> failure)
- `low_ccs_reliance`: cumulative CCS (2020-2100) at or below a fixed 1000 Gt CO2 threshold
  (~the ensemble median); scenarios with no reported cumulative CCS can't be confirmed low-reliance
  and are excluded from the desired region rather than counted as low
- `desired_success`: both - reaches net zero early *without* leaning heavily on CCS

In [49]:
CCS_THRESHOLD = 1000  # Gt CO2, fixed cut (~ensemble median of cumulative CCS)

merged["success_nz2070"] = merged[NZ_COL].le(2070)
merged["low_ccs_reliance"] = merged[CCS_COL] <= CCS_THRESHOLD
merged["desired_success"] = merged["success_nz2070"] & merged["low_ccs_reliance"]

print(f"Missing cumulative CCS: {merged[CCS_COL].isna().sum()} scenarios")
print(pd.crosstab(merged["success_nz2070"], merged["low_ccs_reliance"], dropna=False, margins=True))
print(f"\ndesired_success (NZ<=2070 AND low CCS): {merged['desired_success'].sum()} scenarios "
      f"({merged['desired_success'].mean():.1%} of ensemble)")

Missing cumulative CCS: 90 scenarios
low_ccs_reliance  False  True   All
success_nz2070                     
False               147   955  1102
True                125   372   497
All                 272  1327  1599

desired_success (NZ<=2070 AND low CCS): 372 scenarios (23.3% of ensemble)


## 6. Sanity check: factor coverage at the 2030 and 2050 snapshots

Notebook 03 runs PRIM/CART on the 2030 and on the 2050 value of each factor (intersection of
non-missing values per year), so it's worth seeing how many scenarios survive `dropna` at each year.
GDP per capita is the sparsest input, so it sets the joint sample size.

In [ ]:
FACTORS = [
    "Emissions|CO2",
    "Primary Energy|Fossil",
    "Primary Energy|Biomass",
    "Primary Energy|Non-Biomass Renewables",
    "Carbon Capture|Geological Storage|Biomass",
    "Final Energy|Electricity",
    "Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]",
    "Electrification",
    "GDP|PPP",               # numerator for GDP per capita
    "Population",  
    "GDP per capita|PPP",
    "Gross Emissions|CO2|AFOLU"
]


for year in [2040, 2050]:
    cols = [f"{v}|{year}" for v in FACTORS]
    complete = merged[cols].dropna()
    print(f"--- {year}: {len(complete)} scenarios complete on all {len(FACTORS)} factors ---")
    for v in FACTORS:
        col = f"{v}|{year}"
        print(f"  {col[:62]:62s} missing={merged[col].isna().mean():5.1%}")
    print()

--- 2040: 1120 scenarios complete on all 11 factors ---
  Emissions|CO2|2040                                             missing= 0.9%
  Primary Energy|Fossil|2040                                     missing= 1.5%
  Primary Energy|Biomass|2040                                    missing= 1.3%
  Primary Energy|Non-Biomass Renewables|2040                     missing= 8.7%
  Carbon Capture|Geological Storage|Biomass|2040                 missing= 8.9%
  Final Energy|Electricity|2040                                  missing= 2.6%
  Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7 missing= 3.6%
  Electrification|2040                                           missing= 2.6%
  GDP|PPP|2040                                                   missing=14.5%
  Population|2040                                                missing= 9.3%
  GDP per capita|PPP|2040                                        missing=18.4%

--- 2050: 1120 scenarios complete on all 11 factors ---
  Emissions|CO2|20

## 7. Save

In [51]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {merged.shape} to {OUTPUT_CSV}")

Saved (1599, 302) to data_for_scenariodiscovery_full.csv
